In [3]:
!pip install requests beautifulsoup4 lxml fake-useragent tqdm

In [5]:
import requests
from bs4 import BeautifulSoup
from time import sleep
from tqdm import tqdm
import pandas as pd
import re

In [6]:
url = 'https://auto.drom.ru/audi/all/'
page = requests.get(url)
page

<Response [200]>

In [20]:
soup = BeautifulSoup(page.text, 'lxml') #тут смотрю структуру
print(soup.prettify())

<!DOCTYPE html>
<html class="drom-notouch" lang="ru" xml:lang="ru" xmlns="http://www.w3.org/1999/xhtml">
 <head>
  <title>
   Купить Ауди в России: продажа Audi с пробегом и новых от 17 700 рублей.
  </title>
  <meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
  <meta content="drom.ru" name="copyright"/>
  <meta content="telephone=no" name="format-detection"/>
  <meta content="#000000" name="theme-color"/>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta charset="utf-8"/>
  <meta content='{"cf":{"p":{"min":"200000","max":"9750000"},"y":{"min":"1991","max":"2026"},"v":{"min":"1400","max":"4200"},"f":31,"category_id":1},"geor":77,"geoc":1,"id":27,"b":1,"bc":1,"charset":"utf-8"}' name="candy.config"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="13 538 объявлений о продаже Ауди б/у и новых в России от 17 700 рублей – частные объявления. Узнать стоимость Audi и купить с пробегом на Дроме" name="description"/>
  <meta content="ав

In [26]:
links = [] # ищу ссылки на объявления

for a in soup.find_all('a'):
    href = a.get('href')
    if href and re.match(r'https?://auto\.drom\.ru/.+/.+/.+/\d+\.html', href): # match показывает что ссылка подходит под наш паттерн
        links.append(href)

links = list(set(links)) #убираю дубликаты
links[:1000]

['https://auto.drom.ru/moscow/audi/a4/102813737.html',
 'https://auto.drom.ru/ekaterinburg/audi/a6/665811532.html',
 'https://auto.drom.ru/moscow/audi/q5/596509490.html',
 'https://auto.drom.ru/moscow/audi/a8/732297298.html',
 'https://auto.drom.ru/moscow/audi/a7/710480443.html',
 'https://auto.drom.ru/ekaterinburg/audi/q3/470145595.html',
 'https://auto.drom.ru/moscow/audi/a1/574598269.html',
 'https://auto.drom.ru/moscow/audi/a6/217602654.html',
 'https://auto.drom.ru/habarovsk/audi/a3/738096062.html',
 'https://auto.drom.ru/moscow/audi/q5/332859885.html',
 'https://auto.drom.ru/vidnoe/audi/q5/179337071.html',
 'https://auto.drom.ru/moscow/audi/q7/549980325.html',
 'https://auto.drom.ru/moscow/audi/q7/758275176.html',
 'https://auto.drom.ru/spb/audi/q5/405910569.html',
 'https://auto.drom.ru/moscow/audi/a4/162672695.html',
 'https://auto.drom.ru/moscow/audi/s4/480600179.html',
 'https://auto.drom.ru/izhevsk/audi/q3/517369960.html',
 'https://auto.drom.ru/ufa/audi/q3/738450225.html',


In [7]:
from fake_useragent import UserAgent
fake_user = UserAgent()
headers = {'User-Agent': fake_user.random}

In [54]:
url0 = links[1]
page0 = requests.get(url0, headers = headers) #это чтобы рандомные параметры браузера были,чтобы не блочило
soup0 = BeautifulSoup(page0.text, 'lxml') #беру первое объявление и смотрю что внутри
text = soup0.get_text(separator='\n', strip = True) #это чтобы с новой строки были данные и стрип, чтобы пробелов лишних не было, а то там проблемы огромные
print(text[:3000])

Продажа Ауди А6 15 года в Екатеринбурге, Продам Ауди а6 2015г, обмен на более дорогую, комплектация 2.8 FSI quattro S tronic Business, коробка AT, полный привод
Продать
Автомобили
Запчасти
Продажа шин
Отзывы
Каталог
Аукционы Японии
Автомобили из Китая
Автомобили из Кореи
Автомобили из Германии
Электромобили
Каталог китайских авто
ОСАГО онлайн
Автокредиты
Проверка по VIN
Оценить автомобиль
Форумы
ПДД онлайн
Вопросы и ответы
Рейтинг авто
Каталог шин
Договор купли-продажи
Правовые вопросы
Карта сайта
Размещение на Дроме
Разместить прайс
Помощь
Сделка с Дромом на 3 000 000 ₽
Дром
Продажа авто в Екатеринбурге
Audi
A6
Объявление 665811532
Audi A6 2015 в Екатеринбурге
1
/
18
2 200 000
₽
высокая цена
В
кредит
от
39 353
₽
в месяц
Двигатель
бензин, 2.8 л
Мощность
220
л.с.
,
налог
Коробка передач
робот
Привод
4WD
Тип кузова
седан
Цвет
белый
Пробег
238 300
км
Владельцы
3
Руль
левый
Поколение
4 поколение, рестайлинг
Комплектация
2.8 FSI quattro S tronic Business
Отчет по VIN-коду
WAU**************


In [9]:
def GetCar(url0):
    page0 = requests.get(url0, headers = headers)
    soup0 = BeautifulSoup(page0.text, 'lxml')
    car = {}
    car['url'] = url0
    try:
        car['title'] = soup0.find('h1').get_text(strip=True)
    except AttributeError:
        car['title'] = None
    if car['title']:
        year = re.findall(r'\d{4}', car['title'])
        if year:
            car['year'] = int(year[0]) #беру год из заголовка

    #тут марка, модель и город из ссылки
    parts = url0.split('/')
    car['city'] = parts[3]
    car['make'] = parts[4]
    car['model'] = parts[5]

    #дальше работаю с текстом страницы построчно
    text = soup0.get_text('\n', strip=True)
    lines = text.split('\n')

    car['price'] = None
    for line in lines:
        s = line.replace(' ', '').replace('\xa0', '')
        if s.isdigit():
            if 50000 < int(s) < 100000000:
                car['price'] = int(s)
                break

    #еще характеристики
    car['engine'] = None
    car['transmission'] = None
    car['mileage'] = None

    for i in range(len(lines) - 1):
        key = lines[i].lower().strip()
        value = lines[i+1].strip() #тут инфа о параметре на строчку ниже его
        if key == 'двигатель':
            car['engine'] = value
        if key == 'коробка передач':
            car['transmission'] = value
        if key == 'пробег':
            km = re.findall(r'\d+', value.replace(' ', '').replace('\xa0', ''))
            if km:
                car['mileage'] = int(km[0])
    return car

In [74]:
test = GetCar('https://auto.drom.ru/moscow/audi/a4/162672695.html')
test

{'url': 'https://auto.drom.ru/moscow/audi/a4/162672695.html',
 'title': 'Audi A4 2001 в Москве',
 'year': 2001,
 'city': 'moscow',
 'make': 'audi',
 'model': 'a4',
 'price': 850000,
 'engine': 'бензин, 2.0 л',
 'transmission': 'вариатор',
 'mileage': 177000}

In [11]:
def GetCar(url0):
    page0 = requests.get(url0, headers = headers)
    soup0 = BeautifulSoup(page0.text, 'lxml')
    car = {}
    car['url'] = url0
    try:
        car['title'] = soup0.find('h1').get_text(strip=True)
    except AttributeError:
        car['title'] = None
    if car['title']:
        year = re.findall(r'\d{4}', car['title'])
        if year:
            car['year'] = int(year[0])

    #тут марка, модель и город из ссылки
    parts = url0.split('/')
    car['city'] = parts[3]
    car['make'] = parts[4]
    car['model'] = parts[5]

    #дальше работаю с текстом страницы построчно
    text = soup0.get_text('\n', strip=True)
    lines = text.split('\n')

    car['price'] = None
    for line in lines:
        s = line.replace(' ', '').replace('\xa0', '')
        if s.isdigit():
            if 50000 < int(s) < 100000000:
                car['price'] = int(s)
                break

    #еще характеристики
    car['engine'] = None
    car['transmission'] = None
    car['mileage'] = None
    car['drive'] = None
    car['body'] = None
    car['color'] = None
    car['wheel'] = None
    car['hp'] = None

    for i in range(len(lines) - 1):
        key = lines[i].lower().strip()
        value = lines[i+1].strip()
        if key == 'двигатель':
            car['engine'] = value
        if key == 'мощность':
            car['hp'] = int(value)
        if key == 'коробка передач':
            car['transmission'] = value
        if key == 'привод':
            car['drive'] = value
        if key == 'тип кузова':
            car['body'] = value
        if key == 'цвет':
            car['color'] = value
        if key == 'руль':
            car['wheel'] = value
        if key == 'пробег':
            km = re.findall(r'\d+', value.replace(' ', '').replace('\xa0', ''))
            if km:
                car['mileage'] = int(km[0])

    #описание продавца
    car['description'] = None
    for i in range(len(lines)):
        if 'дополнительно' in lines[i].lower():
            car['description'] = ' '.join(lines[i+1:i+20])
            break

    return car

In [13]:
test = GetCar('https://auto.drom.ru/habarovsk/toyota/land_cruiser_prado/921709892.html')
test

{'url': 'https://auto.drom.ru/habarovsk/toyota/land_cruiser_prado/921709892.html',
 'title': 'Ð¡ Ð²Ð°Ñ\x88ÐµÐ³Ð¾ IP-Ð°Ð´Ñ\x80ÐµÑ\x81Ð°Ñ\x81Ð¾Ð²ÐµÑ\x80Ñ\x88ÐµÐ½Ð¾ Ñ\x81Ð»Ð¸Ñ\x88ÐºÐ¾Ð¼Ð¼Ð½Ð¾Ð³Ð¾ Ð·Ð°Ð¿Ñ\x80Ð¾Ñ\x81Ð¾Ð²',
 'city': 'habarovsk',
 'make': 'toyota',
 'model': 'land_cruiser_prado',
 'price': None,
 'engine': None,
 'transmission': None,
 'mileage': None,
 'drive': None,
 'body': None,
 'color': None,
 'wheel': None,
 'hp': None,
 'description': None}

In [86]:
def get_links(make, p):
    url = f'https://auto.drom.ru/{make}/all/page{p}/'
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'lxml')
    
    links = []
    for a in soup.find_all('a'):
        href = a.get('href')
        if href and re.match(r'https?://auto\.drom\.ru/.+/.+/.+/\d+\.html', href):
            links.append(href)
    
    return list(set(links))

In [88]:
test_links = get_links('toyota', 1)
print(len(test_links))
test_links[:5]

26


['https://auto.drom.ru/moscow/toyota/sienta/239292244.html',
 'https://auto.drom.ru/moscow/toyota/highlander/336808218.html',
 'https://auto.drom.ru/moscow/toyota/tacoma/173302475.html',
 'https://auto.drom.ru/moscow/toyota/land_cruiser/390458277.html',
 'https://auto.drom.ru/moscow/toyota/tundra/288523432.html']

In [39]:
makes_batch = ['toyota', 'hyundai', 'kia']
links_1 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_1.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_1))}).to_csv('batch_1.csv', index=False)
print(f'пачка 1: {len(set(links_1))}')

собираю toyota


100%|███████████████████████████████████████████| 15/15 [01:30<00:00,  6.03s/it]


собираю hyundai


100%|███████████████████████████████████████████| 15/15 [01:25<00:00,  5.70s/it]


собираю kia


100%|███████████████████████████████████████████| 15/15 [01:32<00:00,  6.16s/it]

пачка 1: 915


In [41]:
makes_batch = ['volkswagen', 'nissan', 'honda']
links_2 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_2.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_2))}).to_csv('batch_2.csv', index=False)
print(f'пачка 2: {len(set(links_2))}')

собираю volkswagen


100%|███████████████████████████████████████████| 15/15 [01:31<00:00,  6.10s/it]


собираю nissan


100%|███████████████████████████████████████████| 15/15 [01:30<00:00,  6.03s/it]


собираю honda


100%|███████████████████████████████████████████| 15/15 [01:29<00:00,  5.98s/it]

пачка 2: 639


In [42]:
makes_batch = ['mazda', 'bmw', 'ford']
links_3 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_3.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_3))}).to_csv('batch_3.csv', index=False)
print(f'пачка 3: {len(set(links_3))}')

собираю mazda


100%|███████████████████████████████████████████| 15/15 [01:20<00:00,  5.34s/it]


собираю bmw


100%|███████████████████████████████████████████| 15/15 [01:27<00:00,  5.84s/it]


собираю ford


100%|███████████████████████████████████████████| 15/15 [01:28<00:00,  5.91s/it]

пачка 3: 0


In [43]:
makes_batch = ['lada', 'mitsubishi', 'renault']
links_4 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_4.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_4))}).to_csv('batch_4.csv', index=False)
print(f'пачка 4: {len(set(links_4))}')

собираю lada


100%|███████████████████████████████████████████| 15/15 [01:26<00:00,  5.74s/it]


собираю mitsubishi


100%|███████████████████████████████████████████| 15/15 [01:22<00:00,  5.53s/it]


собираю renault


100%|███████████████████████████████████████████| 15/15 [01:23<00:00,  5.57s/it]

пачка 4: 0


In [44]:
makes_batch = ['skoda', 'haval', 'chery']
links_5 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_5.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_5))}).to_csv('batch_5.csv', index=False)
print(f'пачка 5: {len(set(links_5))}')

собираю skoda


100%|███████████████████████████████████████████| 15/15 [01:21<00:00,  5.45s/it]


собираю haval


100%|███████████████████████████████████████████| 15/15 [01:21<00:00,  5.44s/it]


собираю chery


100%|███████████████████████████████████████████| 15/15 [01:20<00:00,  5.34s/it]

пачка 5: 0


In [114]:
b1 = pd.read_csv('batch_1.csv')
b2 = pd.read_csv('batch_2.csv')
b3 = pd.read_csv('batch_3.csv')
b4 = pd.read_csv('batch_4.csv')
b5 = pd.read_csv('batch_5.csv')
all_df = pd.concat([b1, b2, b3, b4, b5])
all_links = all_df['url'].tolist()
pd.DataFrame({'url': all_links}).to_csv('all_links.csv', index=False)

In [116]:
all_links = list(set(all_links))
print(f'собрано: {len(all_links)}')

собрано: 4584


In [31]:
cars = []

for i, link in enumerate(tqdm(all_links)):
    try:
        car = GetCar(link)
        cars.append(car)
        sleep(3)
    except:
        print(link)
    if (i+1) % 200 == 0:  #сохраняю каждые 200 на всякий случай
        df = pd.DataFrame(cars)
        df.to_csv(f'checkpoint_{i+1}.csv', index=False)

100%|█████████████████████████████████████| 6332/6332 [5:42:35<00:00,  3.25s/it]


In [94]:
df = pd.DataFrame(cars)
print(df.shape)

NameError: name 'cars' is not defined

In [118]:
df_links = pd.read_csv('all_links_6_13.csv')
all_links = df_links['url'].tolist()

In [120]:
links_1 = all_links[0:1266]
links_2 = all_links[1266:2532]
links_3 = all_links[2532:3798]
links_4 = all_links[3798:5064]
links_5 = all_links[5064:6332]

In [122]:
pd.DataFrame({'url': links_1}).to_csv('links_1.csv', index=False)
pd.DataFrame({'url': links_2}).to_csv('links_2.csv', index=False)
pd.DataFrame({'url': links_3}).to_csv('links_3.csv', index=False)
pd.DataFrame({'url': links_4}).to_csv('links_4.csv', index=False)
pd.DataFrame({'url': links_5}).to_csv('links_5.csv', index=False)

In [128]:
len(links_4)

1266

In [146]:
import random

In [1]:
links_1_cars = []

for i, link in enumerate(tqdm(links_1)):
    try:
        car = GetCar(link)
        links_1_cars.append(car)
        sleep(random.uniform(1, 7))
    except:
        print(link)
    if (i+1) % 100 == 0:  #сохраняю каждые 100 на всякий случай
        df_links_1 = pd.DataFrame(links_1_cars)
        df_links_1.to_csv(f'links_1_checkpoint_{i+1}.csv', index=False)

NameError: name 'tqdm' is not defined

In [150]:
links_1_cars

[{'url': 'https://auto.drom.ru/novosibirsk/mercedes-benz/gla-class/705394496.html',
  'title': 'Mercedes-Benz GLA-Class 2015 в Новосибирске',
  'year': 2015,
  'city': 'novosibirsk',
  'make': 'mercedes-benz',
  'model': 'gla-class',
  'price': 1599000,
  'engine': 'бензин, 1.6 л',
  'transmission': 'робот',
  'mileage': 93250,
  'drive': 'передний',
  'body': None,
  'color': 'белый',
  'wheel': 'правый',
  'hp': 122,
  'description': ': Для тех кто хочет прикоснуться к настоящему немецкому качеству и Luxury. Продаю самый младший кроссовер Мерседес. Прибыл в Россию в январе 2025 года. Свободная продажа по договору купли-продажи. Оценка японского аукциона 4.5. Отличное состояние, ни одного крашенного элемента, ходовая без замечаний, двигатель масло не кушает. В автомобиле есть почти все современные "навороты". Очень редкий коричневый салон в отличном состоянии. 2 регистратора( перед, зад). Чёрный потолок. Приехал в Новосибирск автовозом. На зимней резине, летнюю тоже отдам. 2 ключа. Ме